In [0]:
%pip install openpyxl

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 01 - Bronze: Ingestão das Fontes Brutas
# MAGIC
# MAGIC Objetivo:
# MAGIC - Ler os arquivos da pasta sources
# MAGIC - Preservar os dados em formato Delta
# MAGIC - Adicionar metadados técnicos de ingestão
# MAGIC - Criar uma base rastreável para as camadas Silver e Gold

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd

In [0]:
BASE_PATH = "/Volumes/case/default/case_data"

SOURCE_PATH = f"{BASE_PATH}/sources"
BRONZE_PATH = f"{BASE_PATH}/bronze"

print("SOURCE_PATH:", SOURCE_PATH)
print("BRONZE_PATH:", BRONZE_PATH)

display(dbutils.fs.ls(SOURCE_PATH))

SOURCE_PATH: /Volumes/case/default/case_data/sources
BRONZE_PATH: /Volumes/case/default/case_data/bronze


path,name,size,modificationTime
dbfs:/Volumes/case/default/case_data/sources/atendimento_ocorrencias.ndjson,atendimento_ocorrencias.ndjson,39129,1781139762000
dbfs:/Volumes/case/default/case_data/sources/cadastro_produtos_api_dump.json,cadastro_produtos_api_dump.json,28302,1781139763000
dbfs:/Volumes/case/default/case_data/sources/comercial_canais.xlsx,comercial_canais.xlsx,5180,1781139764000
dbfs:/Volumes/case/default/case_data/sources/crm_clientes_export.xlsx,crm_clientes_export.xlsx,16938,1781139765000
dbfs:/Volumes/case/default/case_data/sources/erp_pedidos_cabecalho_2025.csv,erp_pedidos_cabecalho_2025.csv,56007,1781139766000
dbfs:/Volumes/case/default/case_data/sources/erp_pedidos_itens_2025.csv,erp_pedidos_itens_2025.csv,40014,1781139767000
dbfs:/Volumes/case/default/case_data/sources/legado_regioes_pipe.txt,legado_regioes_pipe.txt,305,1781139768000
dbfs:/Volumes/case/default/case_data/sources/logistica_entregas.json,logistica_entregas.json,123545,1781139769000
dbfs:/Volumes/case/default/case_data/sources/vendedores.csv,vendedores.csv,1752,1781139770000


In [0]:
def adicionar_metadados_bronze(df, nome_fonte, arquivo_origem):
    """
    Adiciona colunas técnicas para rastreabilidade da ingestão Bronze.
    """
    return (
        df
        .withColumn("_data_ingestao", F.current_timestamp())
        .withColumn("_nome_fonte", F.lit(nome_fonte))
        .withColumn("_arquivo_origem", F.lit(arquivo_origem))
        .withColumn("_camada", F.lit("bronze"))
    )

In [0]:
pedidos_cabecalho_raw = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("inferSchema", True)
    .csv(f"{SOURCE_PATH}/erp_pedidos_cabecalho_2025.csv")
)

pedidos_itens_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{SOURCE_PATH}/erp_pedidos_itens_2025.csv")
)

vendedores_raw = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("inferSchema", True)
    .csv(f"{SOURCE_PATH}/vendedores.csv")
)

regioes_raw = (
    spark.read
    .option("header", True)
    .option("sep", "|")
    .option("inferSchema", True)
    .csv(f"{SOURCE_PATH}/legado_regioes_pipe.txt")
)

produtos_raw = (
    spark.read
    .option("multiLine", True)
    .json(f"{SOURCE_PATH}/cadastro_produtos_api_dump.json")
)

entregas_raw = (
    spark.read
    .option("multiLine", True)
    .json(f"{SOURCE_PATH}/logistica_entregas.json")
)

ocorrencias_raw = (
    spark.read
    .json(f"{SOURCE_PATH}/atendimento_ocorrencias.ndjson")
)

clientes_pdf = pd.read_excel(f"{SOURCE_PATH}/crm_clientes_export.xlsx")
canais_pdf = pd.read_excel(f"{SOURCE_PATH}/comercial_canais.xlsx")

clientes_raw = spark.createDataFrame(clientes_pdf.astype(str))
canais_raw = spark.createDataFrame(canais_pdf.astype(str))

No such comm: LSP_COMM_ID
No such comm: LSP_COMM_ID


In [0]:
fontes_bronze = {
    "pedidos_cabecalho": adicionar_metadados_bronze(
        pedidos_cabecalho_raw,
        "erp_pedidos_cabecalho",
        "erp_pedidos_cabecalho_2025.csv"
    ),
    "pedidos_itens": adicionar_metadados_bronze(
        pedidos_itens_raw,
        "erp_pedidos_itens",
        "erp_pedidos_itens_2025.csv"
    ),
    "vendedores": adicionar_metadados_bronze(
        vendedores_raw,
        "vendedores",
        "vendedores.csv"
    ),
    "regioes": adicionar_metadados_bronze(
        regioes_raw,
        "legado_regioes",
        "legado_regioes_pipe.txt"
    ),
    "produtos": adicionar_metadados_bronze(
        produtos_raw,
        "cadastro_produtos_api",
        "cadastro_produtos_api_dump.json"
    ),
    "entregas": adicionar_metadados_bronze(
        entregas_raw,
        "logistica_entregas",
        "logistica_entregas.json"
    ),
    "ocorrencias": adicionar_metadados_bronze(
        ocorrencias_raw,
        "atendimento_ocorrencias",
        "atendimento_ocorrencias.ndjson"
    ),
    "clientes": adicionar_metadados_bronze(
        clientes_raw,
        "crm_clientes",
        "crm_clientes_export.xlsx"
    ),
    "canais": adicionar_metadados_bronze(
        canais_raw,
        "comercial_canais",
        "comercial_canais.xlsx"
    )
}

# Gravação das fontes brutas em Delta Lake na camada Bronze.
# Nesta camada, os dados são preservados próximos ao formato original,
# acrescidos apenas de metadados técnicos para rastreabilidade.

In [0]:
for nome_tabela, df in fontes_bronze.items():
    caminho_delta = f"{BRONZE_PATH}/{nome_tabela}"
    
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(caminho_delta)
    )
    
    print(f"Tabela Bronze gravada: {nome_tabela} | Linhas: {df.count()} | Caminho: {caminho_delta}")

Tabela Bronze gravada: pedidos_cabecalho | Linhas: 403 | Caminho: /Volumes/case/default/case_data/bronze/pedidos_cabecalho
Tabela Bronze gravada: pedidos_itens | Linhas: 995 | Caminho: /Volumes/case/default/case_data/bronze/pedidos_itens
Tabela Bronze gravada: vendedores | Linhas: 42 | Caminho: /Volumes/case/default/case_data/bronze/vendedores
Tabela Bronze gravada: regioes | Linhas: 8 | Caminho: /Volumes/case/default/case_data/bronze/regioes
Tabela Bronze gravada: produtos | Linhas: 72 | Caminho: /Volumes/case/default/case_data/bronze/produtos
Tabela Bronze gravada: entregas | Linhas: 322 | Caminho: /Volumes/case/default/case_data/bronze/entregas
Tabela Bronze gravada: ocorrencias | Linhas: 270 | Caminho: /Volumes/case/default/case_data/bronze/ocorrencias
Tabela Bronze gravada: clientes | Linhas: 183 | Caminho: /Volumes/case/default/case_data/bronze/clientes
Tabela Bronze gravada: canais | Linhas: 8 | Caminho: /Volumes/case/default/case_data/bronze/canais


In [0]:
display(dbutils.fs.ls(BRONZE_PATH))

path,name,size,modificationTime
dbfs:/Volumes/case/default/case_data/bronze/canais/,canais/,0,1781141541061
dbfs:/Volumes/case/default/case_data/bronze/clientes/,clientes/,0,1781141541061
dbfs:/Volumes/case/default/case_data/bronze/entregas/,entregas/,0,1781141541061
dbfs:/Volumes/case/default/case_data/bronze/ocorrencias/,ocorrencias/,0,1781141541061
dbfs:/Volumes/case/default/case_data/bronze/pedidos_cabecalho/,pedidos_cabecalho/,0,1781141541061
dbfs:/Volumes/case/default/case_data/bronze/pedidos_itens/,pedidos_itens/,0,1781141541061
dbfs:/Volumes/case/default/case_data/bronze/produtos/,produtos/,0,1781141541061
dbfs:/Volumes/case/default/case_data/bronze/regioes/,regioes/,0,1781141541061
dbfs:/Volumes/case/default/case_data/bronze/vendedores/,vendedores/,0,1781141541061


In [0]:
for nome_tabela in fontes_bronze.keys():
    df_validacao = spark.read.format("delta").load(f"{BRONZE_PATH}/{nome_tabela}")
    
    print("=" * 100)
    print(f"bronze.{nome_tabela}")
    print("Linhas:", df_validacao.count())
    df_validacao.printSchema()
    display(df_validacao.limit(5))

bronze.pedidos_cabecalho
Linhas: 403
root
 |-- order_id: string (nullable = true)
 |-- customer_code: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- promised_date: string (nullable = true)
 |-- status_order: string (nullable = true)
 |-- gross_amount: string (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- net_amount: double (nullable = true)
 |-- payment_details: string (nullable = true)
 |-- last_update: timestamp (nullable = true)
 |-- _data_ingestao: timestamp (nullable = true)
 |-- _nome_fonte: string (nullable = true)
 |-- _arquivo_origem: string (nullable = true)
 |-- _camada: string (nullable = true)



order_id,customer_code,seller_id,order_date,promised_date,status_order,gross_amount,discount_amount,net_amount,payment_details,last_update,_data_ingestao,_nome_fonte,_arquivo_origem,_camada
O00001,C0002,V036,2025-07-31,06/08/2025,Faturado,10788.64,0.0,10788.64,"""{""""priority"""": null, """"source"""": """"APP""""}""",2025-08-20T00:00:00.000Z,2026-06-11T01:31:42.875Z,erp_pedidos_cabecalho,erp_pedidos_cabecalho_2025.csv,bronze
O00002,C0023,V008,2025/04/13,2025-04-26,EM_SEPARACAO,14462.32,0.0,14462.32,"""{""""priority"""": """"low"""", """"source"""": """"ERP""""}""",2025-04-26T00:00:00.000Z,2026-06-11T01:31:42.875Z,erp_pedidos_cabecalho,erp_pedidos_cabecalho_2025.csv,bronze
O00003,C0127,V012,2025/04/01,16/04/2025,faturado,13802.56,1994.35,11808.21,"""{""""priority"""": """"low"""", """"source"""": """"APP""""}""",2025-04-19T00:00:00.000Z,2026-06-11T01:31:42.875Z,erp_pedidos_cabecalho,erp_pedidos_cabecalho_2025.csv,bronze
O00004,C0165,V035,2025/11/05,07/11/2025,cancelado,13933.98,670.75,13263.23,"""{""""priority"""": null, """"source"""": """"APP""""}""",2025-11-11T00:00:00.000Z,2026-06-11T01:31:42.875Z,erp_pedidos_cabecalho,erp_pedidos_cabecalho_2025.csv,bronze
O00005,C0048,V001,2026/02/19,21/02/2026,null,5027.19,0.0,5027.19,"""{""""priority"""": """"high"""", """"source"""": """"Portal""""}""",2026-02-25T00:00:00.000Z,2026-06-11T01:31:42.875Z,erp_pedidos_cabecalho,erp_pedidos_cabecalho_2025.csv,bronze


bronze.pedidos_itens
Linhas: 995
root
 |-- order_id: string (nullable = true)
 |-- item_seq: integer (nullable = true)
 |-- product_code: string (nullable = true)
 |-- quantity: double (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- total_item: double (nullable = true)
 |-- item_status: string (nullable = true)
 |-- _data_ingestao: timestamp (nullable = true)
 |-- _nome_fonte: string (nullable = true)
 |-- _arquivo_origem: string (nullable = true)
 |-- _camada: string (nullable = true)



order_id,item_seq,product_code,quantity,unit_price,total_item,item_status,_data_ingestao,_nome_fonte,_arquivo_origem,_camada
O00001,1,P0027,5.0,453.12,2265.6,Ativo,2026-06-11T01:31:47.370Z,erp_pedidos_itens,erp_pedidos_itens_2025.csv,bronze
O00001,2,P0035,10.0,2278.1,22781.0,null,2026-06-11T01:31:47.370Z,erp_pedidos_itens,erp_pedidos_itens_2025.csv,bronze
O00001,3,P0063,2.0,629.8,1259.6,Ativo,2026-06-11T01:31:47.370Z,erp_pedidos_itens,erp_pedidos_itens_2025.csv,bronze
O00001,4,P0001,3.0,1502.54,4507.62,Ativo,2026-06-11T01:31:47.370Z,erp_pedidos_itens,erp_pedidos_itens_2025.csv,bronze
O00002,1,P0032,3.0,"1274,78",3824.34,Ativo,2026-06-11T01:31:47.370Z,erp_pedidos_itens,erp_pedidos_itens_2025.csv,bronze


bronze.vendedores
Linhas: 42
root
 |-- seller_id: string (nullable = true)
 |-- seller_name: string (nullable = true)
 |-- canal_id: string (nullable = true)
 |-- regional_code: string (nullable = true)
 |-- hire_date: string (nullable = true)
 |-- status: string (nullable = true)
 |-- _data_ingestao: timestamp (nullable = true)
 |-- _nome_fonte: string (nullable = true)
 |-- _arquivo_origem: string (nullable = true)
 |-- _camada: string (nullable = true)



...

created_at,customer_code,event_type,metadata,order_id,severity,status,ticket_id,_data_ingestao,_nome_fonte,_arquivo_origem,_camada
2025-01-04 05:00:00,null,refund,null,O00385,High,closed,T00001,2026-06-11T01:32:01.765Z,atendimento_ocorrencias,atendimento_ocorrencias.ndjson,bronze
2026/02/12,null,troca,null,O00051,medium,open,T00002,2026-06-11T01:32:01.765Z,atendimento_ocorrencias,atendimento_ocorrencias.ndjson,bronze
2026/02/07,null,refund,null,O00181,null,null,T00003,2026-06-11T01:32:01.765Z,atendimento_ocorrencias,atendimento_ocorrencias.ndjson,bronze
2025/12/18,null,delay,null,O00371,low,null,T00004,2026-06-11T01:32:01.765Z,atendimento_ocorrencias,atendimento_ocorrencias.ndjson,bronze
27/02/2025 16:00,null,null,null,O00055,medium,Open,T00005,2026-06-11T01:32:01.765Z,atendimento_ocorrencias,atendimento_ocorrencias.ndjson,bronze


bronze.clientes
Linhas: 183
root
 |-- customer_id: string (nullable = true)
 |-- nome_cliente: string (nullable = true)
 |-- segmento: string (nullable = true)
 |-- porte: string (nullable = true)
 |-- cidade: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- data_cadastro: string (nullable = true)
 |-- email: string (nullable = true)
 |-- status_cliente: string (nullable = true)
 |-- updated_at: string (nullable = true)
 |-- _data_ingestao: timestamp (nullable = true)
 |-- _nome_fonte: string (nullable = true)
 |-- _arquivo_origem: string (nullable = true)
 |-- _camada: string (nullable = true)



customer_id,nome_cliente,segmento,porte,cidade,estado,data_cadastro,email,status_cliente,updated_at,_data_ingestao,_nome_fonte,_arquivo_origem,_camada
C0001,Cliente 1,nan,Grande,Florianópolis,santa catarina,2024-01-26,cliente1@empresa.com,nan,2025-02-22 00:00:00,2026-06-11T01:32:04.096Z,crm_clientes,crm_clientes_export.xlsx,bronze
C0002,Cliente 2,Educação,grande,Curitiba,Paraná,2024-03-30,cliente2@empresa.com,Ativo,2025-04-30 00:00:00,2026-06-11T01:32:04.096Z,crm_clientes,crm_clientes_export.xlsx,bronze
C0003,Cliente 3,Varejo,nan,Curitiba,PR,2025/09/08,cliente3@empresa.com,nan,2025-10-07 00:00:00,2026-06-11T01:32:04.096Z,crm_clientes,crm_clientes_export.xlsx,bronze
C0004,Cliente 4,Saúde,nan,Uberlândia,Minas Gerais,2024-08-13,cliente4@empresa.com,nan,2025-08-05 00:00:00,2026-06-11T01:32:04.096Z,crm_clientes,crm_clientes_export.xlsx,bronze
C0005,Cliente 5,Indústria,Média,Niterói,Rio de Janeiro,2024-10-11,cliente5@empresa.com,Inativo,2025-02-19 00:00:00,2026-06-11T01:32:04.096Z,crm_clientes,crm_clientes_export.xlsx,bronze


bronze.canais
Linhas: 8
root
 |-- id_canal: string (nullable = true)
 |-- nome_canal: string (nullable = true)
 |-- tipo_canal: string (nullable = true)
 |-- ativo: string (nullable = true)
 |-- observacao: string (nullable = true)
 |-- _data_ingestao: timestamp (nullable = true)
 |-- _nome_fonte: string (nullable = true)
 |-- _arquivo_origem: string (nullable = true)
 |-- _camada: string (nullable = true)



id_canal,nome_canal,tipo_canal,ativo,observacao,_data_ingestao,_nome_fonte,_arquivo_origem,_camada
CH01,Inside Sales,Direto,sim,nan,2026-06-11T01:32:06.276Z,comercial_canais,comercial_canais.xlsx,bronze
CH02,Parceiros,Indireto,Sim,nan,2026-06-11T01:32:06.276Z,comercial_canais,comercial_canais.xlsx,bronze
CH03,Marketplace,Digital,SIM,nan,2026-06-11T01:32:06.276Z,comercial_canais,comercial_canais.xlsx,bronze
CH04,Field Sales,Direto,nao,canal legado,2026-06-11T01:32:06.276Z,comercial_canais,comercial_canais.xlsx,bronze
CH05,E-commerce,digital,sim,nan,2026-06-11T01:32:06.276Z,comercial_canais,comercial_canais.xlsx,bronze
